In [1]:
import pandas as pd

def load_participant_data(participant_number):
    """
    Load .csv data for a given participant number using the filename format.

    Parameters:
        participant_number (int or str): The participant number (e.g., 9 for '09')

    Returns:
        pd.DataFrame: The loaded DataFrame, or None if not found.
    """
    # Ensure we have two digits (e.g., 9 => '09')
    participant_str = f"{int(participant_number):02d}"
    # Construct the file path
    file_path = f"D:/LegoVR/unity-lego-vr/Other_than_in_project_files/ET_Data/{participant_str}_ET_Data_2025-08-29.csv"
    try:
        df = pd.read_csv(file_path)
        print(f"Loaded data for participant {participant_str} from '{file_path}'")
        return df
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return None

# Example usage:
df = load_participant_data(9)


Loaded data for participant 09 from 'D:/LegoVR/unity-lego-vr/Other_than_in_project_files/ET_Data/09_ET_Data_2025-08-29.csv'


In [14]:
df.columns

Index(['gaze_capture_time', 'raw_timestamp',
       'relative_to_unix_epoch_timestamp', 'focus_distance', 'frame_number',
       'stability', 'status', 'gaze_forward_x', 'gaze_forward_y',
       'gaze_forward_z', 'gaze_origin_x', 'gaze_origin_y', 'gaze_origin_z',
       'left_forward_x', 'left_forward_y', 'left_forward_z', 'left_origin_x',
       'left_origin_y', 'left_origin_z', 'left_status', 'left_pupil_diameter',
       'left_iris_diameter', 'left_pupil_iris_ratio', 'left_eye_openness',
       'right_forward_x', 'right_forward_y', 'right_forward_z',
       'right_origin_x', 'right_origin_y', 'right_origin_z', 'right_status',
       'right_pupil_diameter', 'right_iris_diameter', 'right_pupil_iris_ratio',
       'right_eye_openness', 'inter_pupillary_distance', 'hmd_position_x',
       'hmd_position_y', 'hmd_position_z', 'hmd_rotation_x', 'hmd_rotation_y',
       'hmd_rotation_z', 'hmd_rotation_w', 'model_name', 'is_building_model',
       'hit_obj_name', 'calibration_state', 'calibr

In [18]:
import numpy as np

filtered_df = df[(df["model_name"] != "TM") & (df["is_building_model"] == True)][["gaze_capture_time", "model_name", "hit_obj_name"]]

# Ensure gaze_capture_time is numeric
filtered_df = filtered_df.copy()
filtered_df["gaze_capture_time"] = pd.to_numeric(filtered_df["gaze_capture_time"], errors="coerce")

# Collect all model streaks results across models
all_streaks = []

# Calculate consecutive Model_plate streaks and durations per model
for model, model_group in filtered_df.groupby("model_name"):
    model_group_sorted = model_group.sort_values("gaze_capture_time").reset_index(drop=True)
    mask = model_group_sorted["hit_obj_name"] == "Model_plate"
    # Mark the start of each new streak of consecutive Model_plate
    streak_id = (mask != mask.shift()).cumsum()
    model_group_sorted["streak_id"] = streak_id

    for streak_num, streak_df in model_group_sorted.groupby("streak_id"):
        if streak_df["hit_obj_name"].iloc[0] == "Model_plate":
            start_time = streak_df["gaze_capture_time"].iloc[0]
            end_time = streak_df["gaze_capture_time"].iloc[-1]
            duration_ns = (end_time - start_time)
            if duration_ns < 0 or np.isnan(duration_ns):
                duration_ns = np.nan
                duration_ms = np.nan
                duration_s = np.nan
            else:
                duration_ms = duration_ns / 1e6
                duration_s = duration_ns / 1e9
            # Each row represents one streak, so num_gaze is 1
            all_streaks.append({
                "model_name": model,
                "start_time": start_time,
                "end_time": end_time,
                "duration_ns": duration_ns,
                "duration_ms": duration_ms,
                "duration_s": duration_s,
                "num_gaze": 1
            })

# Compile all streaks into one DataFrame for all models
model_plate_streaks_df = pd.DataFrame(all_streaks)

# The resulting DataFrame now includes duration_ns, duration_ms, and duration_s for each streak.
display(model_plate_streaks_df)

,model_name,start_time,end_time,duration_ns,duration_ms,duration_s,num_gaze
0,C1M1F,1000001823119220300,1000001824119345500,1000125200,1000.1252,1.000125,1
1,C1M2A,1000002680988494000,1000002680988494000,0,0.0000,0.000000,1
2,C1M2A,1000002680998494000,1000002684759970200,3761476200,3761.4762,3.761476,1
3,C1M3F,1000002806016280700,1000002807181428300,1165147600,1165.1476,1.165148,1
4,C1M4A,1000001063662415400,1000001067773927500,4111512100,4111.5121,4.111512,1
...,...,...,...,...,...,...,...
113,C3M6A,1000002109788400600,1000002111339583600,1551183000,1551.1830,1.551183,1
114,C3M6A,1000002119202578800,1000002119202578800,0,0.0000,0.000000,1
115,C3M6A,1000002119267587700,1000002121242838900,1975251200,1975.2512,1.975251,1
116,C3M6A,1000002125249344600,1000002126270474300,1021129700,1021.1297,1.021130,1
